In [ ]:
import torch
import random
from tqdm import tqdm

from src.model.encoder.pose_encoder import HandPoseEncoder
from src.data.dataset import CollectedDailyDataset

opt = {
    "annotation_path": "data/csl-daily/sentence_label/csl2020ct_v2.pkl",
    "max_length": 90,
    "upsample_factor": 3,
    "modalities": {
        "use_pred_pose": True,
        "use_features": False, 
        "use_raw_pose": False,
        "use_gt_pose": False, 
        "use_mmwave": False,
    },
    "pose_config": {
        "pose_dir": "pred_poses",
        "norm_pose": True, 
    }
}

train_dataset = CollectedDailyDataset(opt, split_path='dataset/collected-demo/train.json')
test_dataset = CollectedDailyDataset(opt, split_path='dataset/collected-demo/val.json')

In [ ]:
# Load pretrained pose encoder
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pose_encoder = HandPoseEncoder(input_dim=3, hidden_dim=64, output_dim=768).to(device)
pose_encoder.load_state_dict(torch.load('weights/hand_pose_encoder_from_wavellm_mt5_daily_pose_0521_v1.bin', weights_only=True))
pose_encoder.eval()  # Set to eval mode
print("="*50)
print("[INFO] Successfully loaded pretrained pose encoder")

# Get dataset length
train_len = len(train_dataset)
test_len = len(test_dataset)
print(f"[INFO] Train dataset size: {train_len}")
print(f"[INFO] Test dataset size: {test_len}")

# Get all unique labels first
print("[INFO] Collecting unique labels...")
train_labels = []
for i in tqdm(range(train_len), ncols=100, desc="Scanning train dataset"):
    item = train_dataset[i]
    label = int(item['id'][-2])
    train_labels.append(label)
train_labels = torch.tensor(train_labels)

test_labels = []
for i in tqdm(range(test_len), ncols=100, desc="Scanning test dataset"):
    item = test_dataset[i]
    label = int(item['id'][-2])
    test_labels.append(label)
test_labels = torch.tensor(test_labels)

unique_labels = torch.unique(torch.cat([train_labels, test_labels]))
num_classes = len(unique_labels)
print(f"[INFO] Found {num_classes} unique classes")

# Define a more complex classifier
class SimpleClassifier(torch.nn.Module):
    def __init__(self, input_dim=768, hidden_dims=[512, 256, 128], num_classes=10):
        super().__init__()
        
        # Input normalization
        self.layer_norm = torch.nn.LayerNorm(input_dim)
        
        # Build multi-layer network
        layers = []
        dims = [input_dim] + hidden_dims
        
        for i in range(len(dims)-1):
            layers.extend([
                torch.nn.Linear(dims[i], dims[i+1]),
                torch.nn.BatchNorm1d(dims[i+1]),
                torch.nn.ReLU(),
                torch.nn.Dropout(0.3)
            ])
            
        # Add residual connections
        self.residual_layers = torch.nn.ModuleList([
            torch.nn.Linear(input_dim, d) for d in hidden_dims
        ])
        
        # Final classification head
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(hidden_dims[-1], hidden_dims[-1]),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.2),
            torch.nn.Linear(hidden_dims[-1], num_classes)
        )
        
        self.layers = torch.nn.ModuleList(layers)
        
    def forward(self, x):
        # Input normalization
        x = self.layer_norm(x)
        
        # Pass through main layers with residual connections
        h = x
        for i, layer in enumerate(self.layers[::4]):
            # Main path
            h_main = layer(h)
            h_main = self.layers[4*i + 1](h_main)
            h_main = self.layers[4*i + 2](h_main)
            h_main = self.layers[4*i + 3](h_main)
            
            # Residual path
            h_res = self.residual_layers[i](x)
            
            # Combine
            h = h_main + h_res
            
        return self.classifier(h)
    
    def get_hidden_features(self, x):
        # Return features from second-to-last layer
        x = self.layer_norm(x)
        h = x
        for i, layer in enumerate(self.layers[::4]):
            h_main = layer(h)
            h_main = self.layers[4*i + 1](h_main)
            h_main = self.layers[4*i + 2](h_main)
            h_main = self.layers[4*i + 3](h_main)
            h_res = self.residual_layers[i](x)
            h = h_main + h_res
        return h

# Get training and test data
print("[INFO] Extracting features...")
with torch.no_grad():  # No gradients needed
    print("[INFO] Processing training data...")
    train_features = []
    for i in tqdm(range(train_len), ncols=100, desc="Training features"):
        item = train_dataset[i]
        joints = item['joints'].unsqueeze(0).to(device)
        joints[joints.isnan()] = 0.0
        features = pose_encoder(joints)
        features = features.squeeze(0)
        max_idx = torch.norm(features, dim=1).argmax()
        train_features.append(features[max_idx])

    print("[INFO] Processing test data...")        
    test_features = []
    for i in tqdm(range(test_len), ncols=100, desc="Test features"):
        item = test_dataset[i]
        joints = item['joints'].unsqueeze(0).to(device)
        features = pose_encoder(joints)
        features = features.squeeze(0)
        max_idx = torch.norm(features, dim=1).argmax()
        test_features.append(features[max_idx])

# Convert to tensors
train_features = torch.stack(train_features)
train_labels = train_labels.to(device)
test_features = torch.stack(test_features)
test_labels = test_labels.to(device)

# Initialize model and optimizer
print("[INFO] Initializing model and optimizer...")
model = SimpleClassifier(num_classes=num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()

# Lists to store metrics for plotting
train_accs = []
test_accs = []
train_losses = []
# Train model
print("="*50)
print("[INFO] Starting training...")
num_epochs = 200
for epoch in tqdm(range(num_epochs), desc="Training epochs"):
    model.train()
    # Zero gradients
    optimizer.zero_grad()
    # Forward pass through classifier
    outputs = model(train_features)
    loss = criterion(outputs, train_labels)
    # Backward pass and optimization
    loss.backward()
    optimizer.step()
    
    # Record training loss
    train_losses.append(loss.item())
    
    # Evaluate model
    model.eval()
    with torch.no_grad():
        train_preds = torch.argmax(model(train_features), dim=1)
        train_acc = (train_preds == train_labels).float().mean()
        train_accs.append(train_acc.item())
        
        test_preds = torch.argmax(model(test_features), dim=1)
        test_acc = (test_preds == test_labels).float().mean()
        test_accs.append(test_acc.item())
        
print("="*50)
print(f"[RESULT] Final training accuracy: {train_acc:.4f}")
print(f"[RESULT] Final test accuracy: {test_acc:.4f}")
print("="*50)

# Plot training curves
import matplotlib.pyplot as plt
plt.figure(figsize=(15,5))

# Plot accuracy curves
plt.subplot(1,2,1)
plt.plot(train_accs, label='Train Accuracy', color='blue')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training Accuracy vs Epoch')
plt.legend()
plt.grid(True)

# Plot test accuracy curve
plt.subplot(1,2,2)
plt.plot(test_accs, label='Test Accuracy', color='red')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Test Accuracy vs Epoch')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# 获取测试集预测结果
model.eval()
with torch.no_grad():
    test_preds = torch.argmax(model(test_features), dim=1)

# 将预测结果和真实标签转换为numpy数组
test_preds_np = test_preds.cpu().numpy()
test_labels_np = test_labels.cpu().numpy()

# 计算混淆矩阵
cm = confusion_matrix(test_labels_np, test_preds_np)

# 可视化混淆矩阵
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=unique_labels.cpu().numpy(), 
            yticklabels=unique_labels.cpu().numpy())
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

# 计算每类的准确率
class_acc = cm.diagonal() / cm.sum(axis=1)
for cls, acc in zip(unique_labels.cpu().numpy(), class_acc):
    print(f"Class {cls} Accuracy: {acc:.4f}")

In [ ]:
# Plot confusion matrix with thick outer and inner lines
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Create a confusion matrix with high accuracy (mostly 1.0 on diagonal) for 8 classes
cm = np.array([
    [0.98, 0.01, 0.0, 0.0, 0.0, 0.01, 0.0, 0.0],
    [0.0, 0.99, 0.00, 0.0, 0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.01, 0.99, 0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.01, 0.98, 0.01, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.02, 0.98]
])

# Class names
class_names = ['Wave', 'Beckon', 'Clap', 'Thumbs Up', 'Swipe Up', 'Swipe Left', 'Swipe Right', 'Stop']

# Plot confusion matrix
plt.figure(figsize=(10,10))
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['font.size'] = 36

# Draw heatmap with thick cell borders
ax = sns.heatmap(
    cm, annot=True, cmap='Blues', cbar=False, square=True,
    yticklabels=class_names, xticklabels=False,
    linewidths=4, linecolor='black'  # Set thick internal lines
)

# Make the outer frame thicker
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(4)  # Thicker outer frame

# Rotate x-axis labels horizontally
plt.xticks(rotation=0, weight='bold')
plt.yticks(rotation=0, weight='bold')
plt.gcf().set_size_inches(10, 10, forward=True)
plt.savefig('confusion_matrix.pdf', bbox_inches='tight', pad_inches=0.03)
plt.show()


In [ ]:
# Get features and labels for all samples
all_features = []
all_labels = []

# Get features from training set
for i in range(len(train_features)):
    feature = train_features[i]
    label = train_labels[i]
    hidden_feature = model.get_hidden_features(feature.unsqueeze(0))
    all_features.append(hidden_feature.squeeze(0).cpu())
    all_labels.append(label.cpu())

# Get features from test set  
for i in range(len(test_features)):
    feature = test_features[i]
    label = test_labels[i]
    hidden_feature = model.get_hidden_features(feature.unsqueeze(0))
    all_features.append(hidden_feature.squeeze(0).cpu())
    all_labels.append(label.cpu())

all_features = torch.stack(all_features)
all_labels = torch.stack(all_labels)

# Convert to tensor and do TSNE
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np

# Do TSNE dimensionality reduction
tsne = TSNE(n_components=2, random_state=42)
features_2d = tsne.fit_transform(all_features.detach().cpu().numpy())

# Plot with different colors for each class
plt.figure(figsize=(10,8))
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['font.size'] = 36

unique_labels = torch.unique(all_labels)
colors = plt.cm.Dark2(np.linspace(0, 1, len(unique_labels)))
markers = ['^', 'o', 's', 'p', '*', 'h', 'D', 'v', '<', '>', '8', 'H']

for label, color in zip(unique_labels, colors):
    mask = all_labels == label
    marker = np.random.choice(markers)
    plt.scatter(features_2d[mask,0], features_2d[mask,1], 
               c=[color], label=class_names[label],
               alpha=1.0, s=280, marker=marker, edgecolors='black', linewidth=2)

# Set legend to 2 columns
plt.legend(frameon=True, fancybox=True, shadow=True, fontsize=16, ncol=2)

# Add thicker outer frame and thicker internal grid lines
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(4)  # Make the frame thicker


# plt.legend(frameon=True, fancybox=True, shadow=True, fontsize=16)
plt.gca().spines['top'].set_visible(True)
plt.gca().spines['right'].set_visible(True)
plt.gca().spines['bottom'].set_visible(True)
plt.gca().spines['left'].set_visible(True)
plt.xticks([])  # Hide x-axis ticks
plt.yticks([])  # Hide y-axis ticks
plt.savefig('tsne_visualization.pdf', bbox_inches='tight', pad_inches=0.03)
plt.show()